# Зависимости + Загрузка данных

In [ ]:
# Установка библиотек 
!pip install catboost mlflow scikit-learn -q

import pandas as pd
import mlflow
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_score

# 1. Загрузка витрины из Parquet
df = pd.read_parquet("/home/jovyan/work/data/features/")
print(f"✅ Загружено {len(df)} строк | Фичи: {list(df.columns)}")

# Препроцессинг + Обучение

In [ ]:
# 2. Создание таргета (упрощённая логика для демо)
df["target_purchase"] = (df["daily_spend"] > 500).astype(int)

# 3. Разделение (временной сплит: без shuffle, чтобы не было leakage)
feature_cols = ["sessions_count", "events_count", "daily_spend", "unique_event_types", "active_transitions"]
X = df[feature_cols].fillna(0)
y = df["target_purchase"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=False)

# 4. Обучение CatBoost
model = CatBoostClassifier(
    iterations=150, depth=5, learning_rate=0.05,
    loss_function="Logloss", eval_metric="AUC", verbose=False, random_seed=42
)
model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=20)
print("✅ Модель обучена")

# Оценка + Логирование

In [ ]:
# 5. Метрики
pred_proba = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, pred_proba)
prec = precision_score(y_test, (pred_proba > 0.5).astype(int), zero_division=0)
print(f"📊 ROC-AUC: {auc:.3f} | Precision@0.5: {prec:.3f}")

# 6. Локальное логирование в MLflow (без внешнего сервера)
mlflow.set_tracking_uri("file:///home/jovyan/work/mlruns")
with mlflow.start_run(run_name="purchase_v1_demo"):
    mlflow.log_params({"iterations": 150, "depth": 5})
    mlflow.log_metrics({"roc_auc": auc, "precision": prec})
    mlflow.catboost.log_model(model, "model")
    print("✅ Артефакты сохранены в ./mlruns/")

# 7. Важность фичей
import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
pd.Series(model.feature_importances_, index=feature_cols).sort_values().plot(kind="barh")
plt.title("Feature Importance (CatBoost)")
plt.tight_layout()
plt.show()